# 07 - Classificador supervisionado leve e seletivo

Treina tres baselines (estruturado, texto e combinado), calibra dois limiares sem olhar os cinco dias finais internos e permite abstencao entre eles.

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
ROOT = Path.cwd(); NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
sys.path.insert(0, str(NB_DIR))
from produtividade_30d import (carregar_fontes, preparar_dataset, dividir_por_dia, preparar_features,
    criar_modelo, probabilidade_i, buscar_limiares, predicao_seletiva, metricas)
OUT = Path(os.environ.get('KV_30D_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_30d')); OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, _ = carregar_fontes(); proxy, _ = preparar_dataset(eventos, catalogo); split = dividir_por_dia(proxy)
treino, calibracao, teste = map(preparar_features, [split.treino, split.calibracao, split.teste_interno])
preds = [] ; resultados = []
for tipo in ['estruturado','texto','combinado']:
    modelo = criar_modelo(tipo); modelo.fit(treino, treino.y_true)
    prob_cal = probabilidade_i(modelo, calibracao)
    escolha = buscar_limiares(calibracao, prob_cal)
    quadros_modelo = []
    for parte, df, prob in [('calibracao', calibracao, prob_cal), ('teste_interno', teste, probabilidade_i(modelo, teste))]:
        pred = predicao_seletiva(prob, escolha['limiar_I'], escolha['limiar_P'])
        resultados.append({'modelo': tipo, 'parte': parte, 'limiar_I': escolha['limiar_I'], 'limiar_P': escolha['limiar_P'], 'passou_calibracao': escolha['passou'], **metricas(df, pred)})
        quadros_modelo.append(pd.DataFrame({'id': df.id.astype(str), 'dia': df.dia, 'split': parte, 'y_true': df.y_true, 'y_baseline': df.y_baseline, f'prob_I_{tipo}': prob}))
    preds.append(pd.concat(quadros_modelo, ignore_index=True))
resultado = pd.DataFrame(resultados)
pred = preds[0]
for quadro in preds[1:]: pred = pred.merge(quadro, on=['id','dia','split','y_true','y_baseline'], how='outer')
resultado.to_csv(OUT / 'modelos_leves_metricas.csv', index=False)
pred.to_csv(OUT / 'modelos_leves_predicoes.csv', index=False)
display(resultado)
print('Nenhum modelo e publicavel apenas por este screening; o holdout continua separado.')